# Count merges and preserve text boundaries

Chapter 19 distinguishes the hand word-boundary BPE, supplied-vocabulary WordPiece encoding and this project's byte BPE. All example text, including localized strings, lives in the shared content/data resources. This implementation explanation is English. No external tokenizer is downloaded.


In [1]:
from pathlib import Path
import json
import sys
import numpy as np
ROOT = Path.cwd()
assert (ROOT / "src/config/book.mjs").is_file(), "Run from the book repository root"
sys.path.insert(0, str(ROOT / "code/part-iv"))
sys.path.insert(0, str(ROOT / "code/mini-gpt"))
from tokenizer_example import run as tokenizer_trace
from tokenizer import load
result = tokenizer_trace()
for row in result["hand_bpe"]:
    print(row["rank"], row["selected"], row["selected_count"], row["segments"])
assert [row["selected_count"] for row in result["hand_bpe"]] == [3, 3, 2, 1]
print("WordPiece", result["wordpiece"])


1 ['l', 'o'] 3 {'low': ['lo', 'w', '</w>'], 'lower': ['lo', 'w', 'e', 'r', '</w>'], 'new': ['n', 'e', 'w', '</w>']}
2 ['lo', 'w'] 3 {'low': ['low', '</w>'], 'lower': ['low', 'e', 'r', '</w>'], 'new': ['n', 'e', 'w', '</w>']}
3 ['low', '</w>'] 2 {'low': ['low</w>'], 'lower': ['low', 'e', 'r', '</w>'], 'new': ['n', 'e', 'w', '</w>']}
4 ['e', 'r'] 1 {'low': ['low</w>'], 'lower': ['low', 'er', '</w>'], 'new': ['n', 'e', 'w', '</w>']}
WordPiece [{'text': 'lower', 'pieces': ['low', '##er']}, {'text': 'lowerz', 'pieces': ['[UNK]']}]


## Bytes, IDs and exact decoding

These are measurements on the same eight strings, not average token lengths for entire languages. The normalization case changes code points, whereas this byte tokenizer performs no normalization.


In [2]:
tokenizer = load(ROOT)
for row in result["roundtrips"]:
    assert tokenizer.decode(row["ids"]) == row["text"]
    print(repr(row["text"]), "codepoints", row["codepoints"], "bytes", row["byte_count"], "tokens", row["token_count"], "IDs", row["ids"])
print("normalization counterexample", result["normalization"])
try:
    tokenizer.decode([255])
except UnicodeDecodeError:
    print("Byte 255 alone: rejected as invalid UTF-8")
else:
    raise AssertionError("Invalid UTF-8 was silently accepted")


'red key .' codepoints 9 bytes 9 tokens 5 IDs [270, 32, 268, 32, 46]
'法国首都是巴黎。' codepoints 8 bytes 24 tokens 24 IDs [230, 179, 149, 229, 155, 189, 233, 166, 150, 233, 131, 189, 230, 152, 175, 229, 183, 180, 233, 187, 142, 227, 128, 130]
'中' codepoints 1 bytes 3 tokens 3 IDs [228, 184, 173]
'🙂' codepoints 1 bytes 4 tokens 4 IDs [240, 159, 153, 130]
'é' codepoints 2 bytes 3 tokens 3 IDs [101, 204, 129]
'é' codepoints 1 bytes 2 tokens 2 IDs [195, 169]
'Ａ A' codepoints 3 bytes 5 tokens 5 IDs [239, 188, 161, 32, 65]
'  red  key .\n' codepoints 13 bytes 13 tokens 9 IDs [32, 32, 270, 32, 32, 268, 32, 46, 10]
normalization counterexample {'raw': 'é', 'normalized': 'é', 'raw_equal': False}
Byte 255 alone: rejected as invalid UTF-8


## Text prefixes need not be token prefixes

The original red diagnostic ends at a colon byte. The aligned probe adds one space and ends with a merged colon-space token. Keep both versions; do not replace the unfavorable original run.


In [3]:
corpus = json.loads((ROOT / "data/mini-gpt/corpus.json").read_text())
probes = json.loads((ROOT / "data/mini-gpt/generation-probes.json").read_text())
original = tokenizer.encode(corpus["diagnostics"][0]["prompt"])
aligned = tokenizer.encode(probes["probes"][0]["prompt"])
full = tokenizer.encode(corpus["train"][4]["text"])
assert original[-1] == 58 and aligned[-1] == 259
assert full[:len(original)] != original
assert full[:len(aligned)] == aligned
print("original IDs", original)
print("aligned IDs", aligned)
print("tokenizer identity", tokenizer.sha256)


original IDs [84, 265, 268, 32, 277, 270, 261, 65, 276, 119, 262, 58]
aligned IDs [84, 265, 268, 32, 277, 270, 261, 65, 276, 119, 262, 259]
tokenizer identity 9bb146ce375bda3aa518a48a60d9f69611b8d0afde9df9a8298eae9dfa9dd354
